In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Load Merged DDoS Dataset
import pandas as pd

path = "/content/drive/My Drive/Cybersecurity_DDoS/final_merged_ddos_400k.csv"
df = pd.read_csv(path)

print(df.head())
print(df.shape)


/tmp/ipython-input-4021928678.py:4: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


   Unnamed: 0                               Flow ID   Source IP   Source Port  \
0       20546  172.16.0.5-192.168.50.1-766-30793-17  172.16.0.5           766   
1       13212  172.16.0.5-192.168.50.1-635-49551-17  172.16.0.5           635   
2       24751  172.16.0.5-192.168.50.1-811-13905-17  172.16.0.5           811   
3        2251  172.16.0.5-192.168.50.1-634-53756-17  172.16.0.5           634   
4       13030  172.16.0.5-192.168.50.1-619-51590-17  172.16.0.5           619   

   Destination IP   Destination Port   Protocol                   Timestamp  \
0    192.168.50.1              30793         17  2018-12-01 11:06:20.702420   
1    192.168.50.1              49551         17  2018-12-01 11:06:15.066238   
2    192.168.50.1              13905         17  2018-12-01 11:06:21.256984   
3    192.168.50.1              53756         17  2018-12-01 11:02:11.873202   
4    192.168.50.1              51590         17  2018-12-01 11:06:25.248014   

    Flow Duration   Total Fwd Packets 

In [ ]:
# Label Encoding (BENIGN = 0, each attack class = 1,2,3,4)
import numpy as np

labels = df[" Label"].unique().tolist()
if "BENIGN" in labels:
    labels.remove("BENIGN")

label_mapping = {"BENIGN": 0}
for i, label in enumerate(sorted(labels), start=1):
    label_mapping[label] = i

df["Label"] = df[" Label"].map(label_mapping).astype(int)

print("Label Mapping:")
print(label_mapping)

print("\nLabel counts:")
print(df["Label"].value_counts())


Label Mapping:
{'BENIGN': 0, 'DrDoS_DNS': 1, 'DrDoS_MSSQL': 2, 'UDP-lag': 3, 'WebDDoS': 4}

Label counts:
Label
1    100000
2    100000
3    100000
0     76674
4       439
Name: count, dtype: int64


In [ ]:
# Keep only numeric features
numeric_df = df.select_dtypes(include=[np.number]).copy()
print("Numeric shape:", numeric_df.shape)


Numeric shape: (377113, 83)


In [ ]:
# Save preprocessed dataset
save_dir = "/content/drive/My Drive/Cybersecurity_DDoS"
preprocessed_path = f"{save_dir}/ddos_preprocessed_data.csv"

numeric_df.to_csv(preprocessed_path, index=False)
print("Saved:", preprocessed_path)


Saved: /content/drive/My Drive/Cybersecurity_DDoS/ddos_preprocessed_data.csv


In [ ]:
# Train/Val/Test Split (60/20/20)
from sklearn.model_selection import train_test_split

X = numeric_df.drop("Label", axis=1)
y = numeric_df["Label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


Train: (226267, 82)
Val: (75423, 82)
Test: (75423, 82)


In [ ]:
# Handle NaN/Inf + Scaling
from sklearn.preprocessing import StandardScaler
import joblib

X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_val.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

X_train = X_train.dropna()
y_train = y_train[X_train.index]

X_val = X_val.dropna()
y_val = y_val[X_val.index]

X_test = X_test.dropna()
y_test = y_test[X_test.index]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, f"{save_dir}/ddos_scaler.pkl")
print("Scaler saved.")


Scaler saved.


In [ ]:
# Reshape for 1D CNN + LSTM
X_train_cnn = X_train_scaled.reshape(len(X_train_scaled), X_train_scaled.shape[1], 1)
X_val_cnn   = X_val_scaled.reshape(len(X_val_scaled),   X_val_scaled.shape[1], 1)
X_test_cnn  = X_test_scaled.reshape(len(X_test_scaled),  X_test_scaled.shape[1], 1)


In [ ]:
# Save Split CSVs
train_df = pd.concat([pd.DataFrame(X_train_scaled), y_train.reset_index(drop=True)], axis=1)
val_df = pd.concat([pd.DataFrame(X_val_scaled), y_val.reset_index(drop=True)], axis=1)
test_df = pd.concat([pd.DataFrame(X_test_scaled), y_test.reset_index(drop=True)], axis=1)

train_df.to_csv(f"{save_dir}/ddos_train_data.csv", index=False)
val_df.to_csv(f"{save_dir}/ddos_val_data.csv", index=False)
test_df.to_csv(f"{save_dir}/ddos_test_data.csv", index=False)


In [ ]:
# Build and Train 1D-CNN + LSTM
import tensorflow as tf
from tensorflow.keras import layers, models

input_shape = (X_train_cnn.shape[1], 1)
num_classes = len(np.unique(y_train))

model = models.Sequential([
    layers.Conv1D(64, 3, activation="relu", padding="same", input_shape=input_shape),
    layers.Conv1D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling1D(pool_size=2),
    layers.LSTM(64, return_sequences=False),
    layers.Dense(128, activation="relu", name="embedding_layer"),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=20,
    batch_size=32
)

model.save(f"{save_dir}/ddos_model_cnn_lstm.h5")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 82, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 82, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 41, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Dense)         │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,333 (325.52 KB)

 Trainable params: 83,333 (325.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 64s 9ms/step - accuracy: 0.9252 - loss: 0.2325 - val_accuracy: 0.9715 - val_loss: 0.1016
Epoch 2/20
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - accuracy: 0.9723 - loss: 0.0981 - val_accuracy: 0.9755 - val_loss: 0.0833
Epoch 3/20
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - accuracy: 0.9759 - loss: 0.0828 - val_accuracy: 0.9775 - val_loss: 0.0851
Epoch 4/20
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 59s 9ms/step - accuracy: 0.9775 - loss: 0.0775 - val_accuracy: 0.9793 - val_loss: 0.0707
Epoch 5/20
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 59s 9ms/step - accuracy: 0.9781 - loss: 0.0744 - val_accuracy: 0.9803 - val_loss: 0.0683
Epoch 6/20
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 59s 9ms/step - accuracy: 0.9796 - loss: 0.0706 - val_accuracy: 0.9794 - val_loss: 0.0707
Epoch 7/20
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 60s 9ms/step - accuracy: 0.9795 - loss: 0.0693 - val_accuracy: 0.9772 - val_loss: 0.0766
Epoch 8/20
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 70s 10ms/step - accuracy: 0.9800 - loss: 

In [ ]:
# Generate Metrics (Train / Val / Test)
from sklearn.metrics import classification_report, confusion_matrix

train_pred = model.predict(X_train_cnn).argmax(axis=1)
val_pred   = model.predict(X_val_cnn).argmax(axis=1)
test_pred  = model.predict(X_test_cnn).argmax(axis=1)

print("\nTrain Report:\n", classification_report(y_train, train_pred))
print("\nValidation Report:\n", classification_report(y_val, val_pred))
print("\nTest Report:\n", classification_report(y_test, test_pred))


6740/6740 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step
2245/2245 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step
2247/2247 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step

Train Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     45529
           1       0.95      1.00      0.97     57250
           2       1.00      0.94      0.97     58601
           3       0.99      1.00      1.00     54033
           4       0.75      0.53      0.62       263

    accuracy                           0.98    215676
   macro avg       0.94      0.89      0.91    215676
weighted avg       0.98      0.98      0.98    215676


Validation Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     15144
           1       0.94      1.00      0.97     19038
           2       1.00      0.94      0.97     19559
           3       0.99      1.00      0.99     17997
           4       0.82      0.47      0.59        88

    accuracy      

In [ ]:
# Build Embedding Extraction Model
from tensorflow.keras import Model, Input

inp = Input(shape=(X_train_cnn.shape[1], 1))
x = inp
for layer in model.layers:
    x = layer(x)
    if layer.name == "embedding_layer":
        break

embedding_output = x
embedding_model = Model(inputs=inp, outputs=embedding_output)

embedding_model.save(f"{save_dir}/ddos_embedding_model_cnn_lstm.h5")


In [ ]:
# Generate and Save Embeddings
train_emb = embedding_model.predict(X_train_cnn)
val_emb   = embedding_model.predict(X_val_cnn)
test_emb  = embedding_model.predict(X_test_cnn)

np.savetxt(f"{save_dir}/ddos_train_embeddings.csv", train_emb, delimiter=",")
np.savetxt(f"{save_dir}/ddos_val_embeddings.csv", val_emb, delimiter=",")
np.savetxt(f"{save_dir}/ddos_test_embeddings.csv", test_emb, delimiter=",")

print("Embedding Extraction Complete!")


6740/6740 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step
2245/2245 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
2247/2247 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step
Embedding Extraction Complete!


In [ ]:
from google.colab import files

# Update these paths if your folder names change
base_path = "/content/drive/My Drive/Cybersecurity_DDoS"

train_path = f"{base_path}/ddos_train_embeddings.csv"
val_path   = f"{base_path}/ddos_val_embeddings.csv"
test_path  = f"{base_path}/ddos_test_embeddings.csv"

print("Downloading embedding files...")

files.download(train_path)
files.download(val_path)
files.download(test_path)

print("Download complete!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete!


In [5]:
# Embedding Extraction Runtime
import time
import numpy as np
import pandas as pd
import tensorflow as tf

base_path = "/content/drive/My Drive/Cybersecurity_DDoS"

# ---------------------------------------------------
# Load embedding model
# ---------------------------------------------------
embedding_model = tf.keras.models.load_model(
    f"{base_path}/ddos_embedding_model_cnn_lstm.h5"
)

print("Model loaded successfully.")
print("Expected input shape:", embedding_model.input_shape)

# ---------------------------------------------------
# Load datasets
# ---------------------------------------------------
train_data = pd.read_csv(f"{base_path}/ddos_train_data.csv")
val_data   = pd.read_csv(f"{base_path}/ddos_val_data.csv")
test_data  = pd.read_csv(f"{base_path}/ddos_test_data.csv")

print("Original train shape:", train_data.shape)

# ---------------------------------------------------
# Remove label column (last column)
# ---------------------------------------------------
X_train = train_data.iloc[:, :-1].values
X_val   = val_data.iloc[:, :-1].values
X_test  = test_data.iloc[:, :-1].values

print("Features shape after label removal:", X_train.shape)

# ---------------------------------------------------
# Reshape for CNN input (82 features → 82x1)
# ---------------------------------------------------
X_train = X_train.reshape(-1, 82, 1)
X_val   = X_val.reshape(-1, 82, 1)
X_test  = X_test.reshape(-1, 82, 1)

print("Final input shape:", X_train.shape)

# ---------------------------------------------------
# Measure embedding extraction time
# ---------------------------------------------------
start_time = time.time()

train_emb = embedding_model.predict(X_train, batch_size=32)
val_emb   = embedding_model.predict(X_val, batch_size=32)
test_emb  = embedding_model.predict(X_test, batch_size=32)

end_time = time.time()

total_time = end_time - start_time
total_samples = len(X_train) + len(X_val) + len(X_test)

print("\n Embedding Extraction Complete!")
print("Total Time:", round(total_time, 2), "seconds")
print("Total Samples:", total_samples)
print("Time per Sample:", round(total_time / total_samples, 6), "seconds")


Model loaded successfully.
Expected input shape: (None, 82, 1)
Original train shape: (215676, 83)
Features shape after label removal: (215676, 82)
Final input shape: (215676, 82, 1)
6740/6740 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step
2245/2245 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step
2247/2247 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step

✅ Embedding Extraction Complete!
Total Time: 34.26 seconds
Total Samples: 359386
Time per Sample: 9.5e-05 seconds
